In [ ]:
import pandas as pd
import numpy as np

# Bias Under Pressure: A Cross-Domain Variance Analysis

> TO DO:

This project investigates whether bias in human judgment under social pressure follows a systematic pattern or reflects random individual error. Using three independent datasets — NBA Last Two Minute reports (2015–2025), the Tegmark media phrase-count corpus, and the BABE expert-annotated sentence dataset — we apply a common variance-based framework across domains where the nature of ground truth differs substantially.

The core measure is error rate asymmetry: the proportion of biased decisions favoring one side over another. Variance of this measure across agents (referee crews, annotators, media outlets) serves as the key discriminator — low variance signals coordinated bias, high variance suggests incidental bias under environmental pressure. Distributions are compared across domains using the Kolmogorov-Smirnov test and bootstrap confidence intervals for the coefficient of variation.

The NBA component provides ground truth validation of the method. The BABE component extends it to expert-annotated text with partial ground truth. The Tegmark component applies it to structural frequency patterns without ground truth. Statistical methods include T-test, ANOVA, Chi-square, Levene test, Logistic Regression, and TF-IDF with Naive Bayes classification.

The project does not claim to prove that bias is measurable in all domains — it investigates whether a method validated on sports officiating transfers to the media domain, and whether variance can distinguish between coordinated manipulation and unconscious social influence.

## Abstract

This project investigates whether bias in human judgment under social pressure follows a systematic pattern or reflects random individual error. Do they all make the same mistake — or different ones?
The core measure is error rate asymmetry: the proportion of biased decisions favouring one side over another. Variance of this measure across agents (referee crews, annotators, media outlets) serves as the key discriminator — low variance signals coordinated bias, high variance suggests incidental bias under environmental pressure. Distributions are compared across domains using the Kolmogorov-Smirnov test and bootstrap confidence intervals for the coefficient of variation.

We start with NBA referees in the last two minutes of close games, where we know what the right call was. We measure how often decisions favour the home team over the visiting team, and how much that varies from one referee crew to the next.

The key idea is this: if every crew leans the same way, something is pushing them. If every crew leans differently, it is probably just noise. Variance tells us which one it is.

We are not trying to prove that bias exists. We are trying to find out whether the math can tell the difference between a coordinated push and an honest mistake.

## Introduction


## Part 1: NBA Last Two Minute Reports

### 1.1 Dataset

The primary dataset is the NBA Last Two Minute (L2M) reports, maintained by atlhawksfanatic [AtlhawksfanaticL2M]. The reports are released by the NBA for games that were within 5 points in the last two minutes, and cover seasons 2015 through 2025.

The dataset contains 90,302 rows and 64 columns. Each row represents a single graded play — either a call made by referees or a potential violation that was not called. The key variable is `decision`, which takes four values: CC (Correct Call), CNC (Correct Non-Call), IC (Incorrect Call), and INC (Incorrect Non-Call). A fifth value — blank — indicates plays that were not detectable without technology and are excluded from analysis.

The dataset is compiled from two sources: archived PDF reports for earlier seasons and the official NBA API for more recent ones. It is supplemented with box score data from stats.nba.com, which adds referee identities, attendance, and player minutes.

Three additional files from the atlhawksfanatic EDA folder are included:
- `haberstroh_ref_ratings.csv` — referee quality ratings by season 
- `haberstroh_team_ratings.csv` — team ratings by season
- `no_minutes.csv` — where `committing_min` is missing due to NBA input errors

The leading reference for this dataset ис [PelechrinisL2M].

#### 1.2 Initial Data Analysis (IDA)

In [4]:
nba_data = pd.read_csv('data/input/nba/tidy/L2M_stats_nba.csv', low_memory=False)

**Shape** — we check the number of rows and columns to confirm the dataset loaded correctly and matches the expected size from the source.

In [8]:
# Shape
print('Shape:', nba_data.shape)

Shape: (90302, 64)


**Data types** — we inspect the type of each column to identify parsing issues. Dates stored as strings or booleans stored as objects will need conversion before analysis.

In [7]:
# Dtypes
print('\nData types:')
print(nba_data.dtypes)


Data types:
period                 object
time                   object
call_type              object
committing             object
disadvantaged          object
                       ...   
disadvantaged_side     object
type2                  object
time_min                int64
time_sec              float64
time2                 float64
Length: 64, dtype: object


**Key columns** — out of 64 columns we focus on the subset relevant to our analysis:

- `decision` — the core variable: CC, CNC, IC, INC, or blank
- `committing_side` — whether the committing player is on the home or away team
- `disadvantaged_side` — whether the disadvantaged player is on the home or away team
- `season` — NBA season (convention: 2015 = 2014-15 season)
- `playoff` — True if the game was a playoff game
- `ATTENDANCE` — game attendance, used as a proxy for crowd pressure
- `OFFICIAL_1`, `OFFICIAL_2`, `OFFICIAL_3` — the three referees for the game
- `call_type` — type of violation or call
- `period` — game period (Q4, Q5 for overtime)
- `time` — time remaining in the period

**Missing values** — we identify columns with missing data early. Some are expected — for example, `committing_min` is missing where the NBA made input errors. Others may signal structural issues in the data.

In [10]:
# Missing values
print('\nMissing values:')
print(nba_data.isnull().sum()[nba_data.isnull().sum() > 0])


Missing values:
call_type                48
committing             1522
disadvantaged          4983
decision               2502
comments                 59
game_details          58270
page                  58270
call                     48
type                     48
networks              88993
game_id               32032
PCTime                32032
ImposibleIndicator    32032
Difficulty            34148
VideolLink            32032
Qualifier             90302
posID                 32032
posStart              32032
posEnd                32032
posTeamId             32032
teamIdInFavor         90302
errorInFavor          90302
imgChart              32032
GameId                32032
GameDate              32032
HomeTeamId            32032
AwayTeamId            32032
L2M_Comments          89809
GAME_ID                 631
OFFICIAL_1              631
OFFICIAL_2              631
OFFICIAL_3              643
OFFICIAL_ID_1           631
OFFICIAL_ID_2           631
OFFICIAL_ID_3           643
OFF

**Missing values** — three categories:

**Expected and structural:**
- `game_details`, `page` — 58,270 missing. These come from the PDF pipeline and are absent for all API-sourced games.
- `networks`, `L2M_Comments`, `Qualifier`, `teamIdInFavor`, `errorInFavor` — largely or entirely missing. Not used in our analysis.
- `OFFICIAL_4`, `OFFICIAL_ID_4` — 88,203 missing. A fourth referee appears only in some playoff games.

**Require attention:**
- `decision` — 2,502 missing. These are blank entries — plays not detectable without technology. Excluded from analysis.
- `committing_side`, `disadvantaged_side` — 2,433 and 5,801 missing. Critical for home/away bias calculation. Will be examined in Tidy.
- `OFFICIAL_1/2/3` — 631–643 missing. Games without referee data cannot be included in referee crew analysis.
- `ATTENDANCE` — 6,509 missing. Affects crowd pressure analysis.

**Known data quality issue:**
- `committing_min` — 3,183 missing. Documented in `no_minutes.csv` — NBA input errors where the player did not actually play.

> **Observation** — missing values fall into three categories: structural (expected from the two-source pipeline), known data quality issues (documented in `no_minutes.csv`), and gaps in key analysis columns (`decision`, `committing_side`, `OFFICIAL_1/2/3`) that will be quantified in Tidy.

**Decision column** — the core variable of the analysis. We examine the distribution of values to understand the balance between correct and incorrect calls, and to confirm the presence of blank entries that will be excluded.

In [11]:
print(nba_data['decision'].value_counts(dropna=False))

decision
CNC    60590
CC     21023
INC     5391
NaN     2502
IC       796
Name: count, dtype: int64


> **Observation** — CNC dominates at 60,590 entries (67%). As noted in [AtlhawksfanaticL2M], CNC criteria are inconsistent across seasons and will be excluded from analysis. NaN (2,502) entries represent plays not detectable without technology and will also be excluded. The working dataset for hypothesis testing will consist of CC, IC, and INC only — 27,210 rows.

**Season column** — we check the distribution of rows by season to confirm coverage and identify any gaps.

In [12]:
print(nba_data['season'].value_counts().sort_index())

season
2015     1972
2016     6864
2017     7532
2018    10583
2019    11408
2020     8222
2021     7917
2022     8859
2023    11502
2024     8278
2025     7165
Name: count, dtype: int64


> **Observation** — the dataset covers 11 seasons from 2015 to 2025. The 2015 season has significantly fewer rows (1,972) as it was the first year the NBA released L2M reports and coverage was limited. Coverage stabilizes from 2016 onward. The COVID seasons (2020–2022) show a moderate drop in row count, consistent with a reduced number of games and the bubble format.

**Playoff column** — we check the split between regular season and playoff games.

In [13]:
print(nba_data['playoff'].value_counts())

playoff
False    83721
True      6581
Name: count, dtype: int64


>**Observation** — 6,581 rows (7.3%) are playoff games. Playoff games will be analyzed separately as [PelechrinisL2M] found home court bias to be more pronounced in the playoffs.

**Committing and disadvantaged side** — these two columns identify whether the committing and disadvantaged player belongs to the home or away team. They are central to the home/away bias calculation.

In [15]:
print('committing_side:')
print(nba_data['committing_side'].value_counts(dropna=False))
print()
print('disadvantaged_side:')
print(nba_data['disadvantaged_side'].value_counts(dropna=False))

committing_side:
committing_side
away    44183
home    43686
NaN      2433
Name: count, dtype: int64

disadvantaged_side:
disadvantaged_side
home    42428
away    42073
NaN      5801
Name: count, dtype: int64


> **Observation** — `committing_side` and `disadvantaged_side` are nearly balanced between home and away, which is expected. Missing values are 2,433 (2.7%) and 5,801 (6.4%) respectively. Rows where either column is missing will be excluded from the home/away bias calculation. The impact of this exclusion will be quantified in Tidy.

**Attendance** — used as a proxy for crowd pressure. We check the range and distribution of values, including games with zero attendance which correspond to the COVID bubble.

In [17]:
print(nba_data[nba_data['ATTENDANCE'] == 0]['season'].value_counts())

season
2022    13
Name: count, dtype: int64


**Unexpected finding** — zero attendance appears only in 2022, not in 2020 as expected from the COVID bubble. We investigate which games these are.

In [19]:
print(nba_data[nba_data['ATTENDANCE'] == 0][['season', 'home_team', 'away_team', 'date']].drop_duplicates())

       season home_team away_team        date
59965    2022   Raptors   Nuggets  2022-02-12


> **Observation** — zero attendance corresponds to a single game: Raptors vs Nuggets on February 12, 2022. This was a game played in Toronto during COVID restrictions that prohibited fans. The NBA bubble (2020) games appear to have attendance recorded as missing rather than zero. This means `ATTENDANCE == 0` is not a reliable indicator of crowd absence — missing values in 2020 are the actual bubble games.

**Referee columns** — each game has up to three referees recorded in `OFFICIAL_1`, `OFFICIAL_2`, `OFFICIAL_3`. A fourth referee appears only in some playoff games.

In [20]:
print('Missing referee data:')
print('OFFICIAL_1:', nba_data['OFFICIAL_1'].isnull().sum())
print('OFFICIAL_2:', nba_data['OFFICIAL_2'].isnull().sum())
print('OFFICIAL_3:', nba_data['OFFICIAL_3'].isnull().sum())
print('OFFICIAL_4:', nba_data['OFFICIAL_4'].isnull().sum())
print()
print('Unique referees (OFFICIAL_1):', nba_data['OFFICIAL_1'].nunique())

Missing referee data:
OFFICIAL_1: 631
OFFICIAL_2: 631
OFFICIAL_3: 643
OFFICIAL_4: 88203

Unique referees (OFFICIAL_1): 100


**Unexpected** — 88,203 missing values for `OFFICIAL_4` seems high. We verify whether the fourth referee appears exclusively in playoff games.

In [21]:
print(nba_data[nba_data['OFFICIAL_4'].notna()]['playoff'].value_counts())

playoff
True     2021
False      78
Name: count, dtype: int64


2,021 out of 2,099 rows with a fourth referee are playoff games, as expected. The remaining 78 regular season games with a fourth referee are likely special cases. This confirms that `OFFICIAL_4` missingness is structural and not a data quality issue.

> **Observation** — referee data is missing for 631–643 rows, representing games where referee identity was not recorded. These rows will be excluded from referee crew analysis. 100 unique referees appear in `OFFICIAL_1`. `OFFICIAL_4` is missing for 88,203 rows — as expected, a fourth referee is rare and appears only in some playoff games.

**Call type** — we check the distribution of violation types to understand what kinds of plays are graded in the L2M reports.

In [22]:
print(nba_data['call_type'].value_counts().head(10))

call_type
Foul: Personal                    27223
Foul: Shooting                    23121
Foul: Offensive                   17318
Foul: Loose Ball                   8303
Turnover: Traveling                3224
Foul: Defense 3 Second             1360
Stoppage: Out-of-Bounds            1092
Instant Replay: Support Ruling      905
Foul: Personal Take                 775
Turnover: 3 Second Violation        704
Name: count, dtype: int64


> **Observation** — foul calls dominate the dataset. Personal fouls (27,223) and shooting fouls (23,121) together account for more than half of all graded plays. This is expected given that fouls are the most common and most contested calls in the final two minutes of close games.

**Period and time** — we check which periods are covered and the time range within each period.

In [23]:
print('Period distribution:')
print(nba_data['period'].value_counts())

Period distribution:
period
Q4    78374
Q5    10533
Q6     1189
Q7      129
Q8       77
Name: count, dtype: int64


> **Observation** — the majority of graded plays occur in Q4 (78,374). Overtime periods Q5 through Q8 account for 11,928 rows (13.2%). Overtime games are included in the analysis as they represent close games where crowd pressure and referee bias are equally relevant.

**Time remaining** — we check the range of time values within periods.

In [24]:
print(nba_data['time_min'].describe())

count    90302.000000
mean         0.435029
std          0.575313
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max         12.000000
Name: time_min, dtype: float64


`time_min` max of 12 is unexpected. L2M reports should only cover the last two minutes of each period. We investigate these rows.

In [29]:
print(nba_data[nba_data['time_min'] > 2][['period', 'time', 'time_min', 'season', 'decision']].head(10))

    period     time  time_min  season decision
6       Q5  04:34.0         4    2015       CC
7       Q5  04:34.0         4    2015       CC
8       Q5  04:05.0         4    2015      CNC
9       Q5  03:54.0         3    2015      INC
134     Q5  04:50.0         4    2015      CNC
135     Q5  04:42.0         4    2015       CC
136     Q5  04:26.0         4    2015      CNC
137     Q5  04:02.0         4    2015       CC
138     Q5  03:43.0         3    2015       CC
139     Q5  03:41.0         3    2015      CNC


Rows with `time_min > 2` are exclusively from Q5 (overtime) in the 2015 season. In overtime, the full 5-minute period was graded, not just the last 2 minutes. This is a known inconsistency in early L2M reporting. These rows are outside the standard 2-minute window but will be retained — excluding them would disproportionately affect 2015 data, which is already the smallest season.

> **Observation** — most graded plays occur in the final minute of the period (median = 0, Q3 = 1). The max of 12 minutes is an anomaly that will be examined in Tidy.

**Supporting files** — we load and briefly inspect the three additional files.

`haberstroh_ref_ratings.csv` — referee quality ratings by season and game type.  
`haberstroh_team_ratings.csv` — team ratings by season and game type.  
`no_minutes.csv` — rows where `committing_min` is missing due to NBA input errors. Used in Tidy to document the exclusion decision.

In [32]:
ref_ratings = pd.read_csv('data/input/nba/eda/haberstroh_ref_ratings.csv')
print('Shape:', ref_ratings.shape)
print(ref_ratings.head())

Shape: (1629, 6)
    szn  szn_type           official  ref_points  games  ref_rating
0  2016  playoffs       Bennie Adams           9      3    3.000000
1  2016  playoffs       Bill Kennedy          22     10    2.200000
2  2016  playoffs       Bill Spooner          11      7    1.571429
3  2016  playoffs        Brian Forte          10      4    2.500000
4  2016  playoffs  Courtney Kirkland           7      3    2.333333


In [33]:
team_ratings = pd.read_csv('data/input/nba/eda/haberstroh_team_ratings.csv')
print('Shape:', team_ratings.shape)
print(team_ratings.head())

Shape: (728, 6)
    szn  szn_type team  ref_rating  tv_games  games
0  2016  playoffs  ATL    2.879471         9     10
1  2016  playoffs  BOS    2.894555         5      6
2  2016  playoffs  CHA    2.759091         5      7
3  2016  playoffs  CLE    3.189198        21     21
4  2016  playoffs  DAL    2.943321         5      5


In [34]:
no_minutes = pd.read_csv('data/input/nba/eda/no_minutes.csv')
print('Shape:', no_minutes.shape)
print(no_minutes.head())

Shape: (40, 50)
  period      time         call_type  committing   disadvantaged decision  \
0     Q4  01:40:00    Foul: Personal  Tony Allen   Ryan Anderson      CNC   
1     Q5  01:40:00    Foul: Personal   Joe Young     Dario Saric      CNC   
2     Q5  00:29:00  Foul: Loose Ball   Joe Young  Ersan Ilyasova      CNC   
3     Q4  00:08:00  Foul: Loose Ball   Joe Young     Enes Kanter      CNC   
4     Q5  00:28:00  Foul: Loose Ball   Joe Young    Steven Adams      CNC   

                                            comments  \
0  Allen (MEM) strips the ball from Asik (NOP) im...   
1  Young (IND) makes incidental contact with Sari...   
2  Young (IND) and Ilyasova (PHI) make incidental...   
3  Young (IND) and Kanter (OKC) briefly engage an...   
4  Young (IND) and Adams (OKC) briefly engage and...   

                          game_details  page                      file  ...  \
0  Grizzlies @ Pelicans (Mar 07, 2015)   1.0    L2M-MEM-NOP-3-7-15.pdf  ...   
1        76ers @ Pacers (N

> **Observation** — all three files loaded correctly.  
> `haberstroh_ref_ratings.csv` covers seasons 2016–2025 with `ref_rating` calculated as `ref_points / games`.
>  A lower rating indicates more errors per game.  
> `haberstroh_team_ratings.csv` covers the same period with team-level ratings based on the quality of referee crews assigned to their games.  
> `no_minutes.csv` has 50 columns — it is a full copy of the main dataset filtered to rows with missing `committing_min`. All 40 rows are CNC decisions, suggesting these are plays where no player was formally identified as committing a violation.

## Reference

[PelechrinisL2M] Pelechrinis, K. (2023). Quantifying implicit biases in refereeing using NBA referees as a testbed. *Scientific Reports*, 13, 4664. https://doi.org/10.1038/s41598-023-31799-y  
[AtlhawksfanaticL2M] atlhawksfanatic. (2025). L2M: NBA Last Two Minute Reports. GitHub. https://github.com/atlhawksfanatic/L2M

